# 04 — NIAH retention curves and the control battery

**Stage:** Proposal Stages 3–4 on a single checkpoint. **Produces:** Table 4 (controls
C1–C4), Table 6 (retention summary), and the raw rows behind Figures 4–7.

### What changed from the pilot

| pilot | here | why |
|---|---|---|
| decoded `ahn_raw` (pre-`o_proj`) | decodes `o_t = o_proj(ahn_raw)` | the pilot's vector was in head-concat space; dims coincide at 3B so it ran and returned noise |
| `vec @ unembed.T` | `readout_logits` with final RMSNorm | Qwen applies `model.norm` before `lm_head` |
| C1 as `o_t(AHN) − o_t(NOWRITE)` ≡ `o_t` | C1 on the **residual stream** | the AHN output is zero under NOWRITE by construction, so the control was vacuous |
| needle at token ~5 | needle placed past `num_attn_sinks` | tokens in the sink prefix are never compressed |
| best layer chosen on the same data it is plotted from | layers fixed in advance | selection-on-test |
| 2 needles (a third silently dropped) | needles filtered up front | cohort size becomes a decision |
| no CIs, no fit diagnostics | bootstrap CIs and R² | Table 6's R² column decides whether "half-life" is even meaningful |

**Production path (8 Sep decision):** set `RUN_CONFIG` once, run the bootstrap/model
cells, then run the corrected **RULER control-battery** cells. Do not Run All: the
homemade-cohort and four-needle sections are retained as historical/supporting analyses.
Notebook 00 and 01 must pass for the selected checkpoint. The shared Qwen2.5-3B J-lens
is required; its original Table 3 failures remain recorded as `lens_validated=False`.


In [1]:
# --- GPU slice: CHANGE THIS EVERY RUN --------------------------------------------
# The H100s are MIG-partitioned: a process gets one ~20 GB slice, not a whole card,
# and the slice UUIDs are regenerated every time the box is recycled (~48h). A bare
# index ("0", "3") does NOT select a MIG slice -- it silently lands somewhere else.
#   nvidia-smi -L    list the slices
#   nvidia-smi       see which are actually idle (four of us share this box)
# Exporting in the shell does not reach the Jupyter kernel, and CUDA reads this once
# at init, so it must be set here -- before anything imports torch.
import os
MIG_UUID = "MIG-802c3ecc-8c66-53d4-9fb5-60712ea8f619"   # <- paste, e.g. "MIG-802c3ecc-8c66-53d4-9fb5-60712ea8f619"
if MIG_UUID:
    os.environ["CUDA_VISIBLE_DEVICES"] = MIG_UUID
elif not os.environ.get("CUDA_VISIBLE_DEVICES"):
    print("! MIG_UUID empty and CUDA_VISIBLE_DEVICES unset -- this kernel lands on "
          "whatever slice it defaults to, possibly one a teammate is using. "
          "Run `nvidia-smi -L`, pick an idle slice, paste its UUID above.")

# --- bootstrap -------------------------------------------------------------------
# `ahn_interp.py` lives at the repo root; this walks up the tree to find it.
# Run this notebook from inside the clone -- nothing needs uploading.
import os, sys, json, importlib

def _find_module(name="ahn_interp.py", depth=4):
    here = os.getcwd()
    for _ in range(depth):
        for cand in (here, os.path.join(here, "src"), os.path.join(here, "notebooks")):
            if os.path.exists(os.path.join(cand, name)):
                return cand
        here = os.path.dirname(here)
    return None

_root = _find_module()
assert _root, ("ahn_interp.py not found. Run this notebook from inside the repo "
               "clone -- `git clone` it on the box rather than copying notebooks "
               "around; the bootstrap searches four levels up from the cwd.")
if _root not in sys.path:
    sys.path.insert(0, _root)

# Pin the working directory to wherever ahn_interp.py actually lives (normally the
# repo root). Without this, relative paths in CFG (results_dir, configs/*.json) resolve
# against whatever directory Jupyter happened to open in -- e.g. running this notebook
# from inside notebooks/ silently writes results to notebooks/results/... instead of
# results/... at the repo root, which is where every other notebook and 05's analysis
# step expect to find them.
os.chdir(_root)

import ahn_interp as ai
importlib.reload(ai)
ai.set_seed()
print("ahn_interp loaded from", _root)
print("working directory pinned to", os.getcwd())


ahn_interp loaded from /home/jupyter-dphs-a263/Interpretability-study-of-Artificial-Hippocampus-Networks
working directory pinned to /home/jupyter-dphs-a263/Interpretability-study-of-Artificial-Hippocampus-Networks


In [ ]:
# --- experiment configuration ----------------------------------------------------
# Select exactly one checked-in config. Change only this line between cell families.
RUN_CONFIG = "run_3b_gdn"  # run_3b_gdn | run_3b_dn | run_3b_m2
ALLOW_OVERWRITE_COMPLETED = False
REQUIRE_JLENS = True

CFG = ai.load_run_config(RUN_CONFIG)
CFG["results_dir"] = os.path.join("results", CFG["run_name"])
os.makedirs(CFG["results_dir"], exist_ok=True)
ai.set_seed(CFG.get("seed", ai.SEED))
ai.set_results_dir(CFG["results_dir"])

def guard_output(name):
    path = os.path.join(CFG["results_dir"], name)
    if os.path.exists(path) and not ALLOW_OVERWRITE_COMPLETED:
        raise FileExistsError(
            f"Refusing to overwrite {path}. Select the intended RUN_CONFIG or set "
            "ALLOW_OVERWRITE_COMPLETED=True for a deliberate replacement."
        )
    return path

print("RUN_CONFIG:", RUN_CONFIG)
print(json.dumps(CFG, indent=2))


In [ ]:
# The J-lens is shared across cells because the Qwen2.5-3B backbone is shared.
# Override this only if the restored files live elsewhere on a fresh GPU box.
JLENS_RESULTS_DIR = os.path.abspath(os.environ.get(
    "AHN_JLENS_RESULTS_DIR", os.path.join("results", "run_3b_gdn")
))

EXP = dict(
    layers              = CFG["layers"],       # fixed in the per-cell run config
    # Prompt length is roughly num_attn_sinks + sliding_window + eviction_distance, so
    # each distance sets the cost of its own conditions: 8192 -> ~16.4K tokens, 16384 ->
    # ~24.6K. Those two dominated the sweep budget. 16384 is dropped from run 1 and added
    # back only if the decay curve has not flattened by 8192 -- six points still support
    # the exponential fit, and Table 6's R2 column is what says whether it does.
    #
    # distance=0 is also dropped: build_niah_prompt puts the needle at ~145 and the
    # compression boundary at n - sliding_window, which for distance=0 lands at ~146, so
    # the ACTUAL eviction distance is ~1 token and the needle_is_evicted check
    # (sinks <= needle_pos < window_start) is one token from failing. Some filler
    # variants would be silently dropped. 64 is the smallest distance that is safely
    # past the boundary for every filler.
    eviction_distances  = [64, 256, 512, 1024, 2048, 4096, 8192],   # add 16384 if needed
    needle_candidates   = ["Paris", "banana", "Tokyo", "violin", "cinnamon",
                           "harbour", "lantern", "sapphire", "meadow", "trumpet"],
    n_filler_variants   = 3,                    # repeats per (needle, distance)
    use_jlens           = True,
    jlens_path          = os.path.join(JLENS_RESULTS_DIR, "jlens_qwen25_3b.pt"),
    jlens_validation_path = os.path.join(JLENS_RESULTS_DIR, "02_table3_jlens_validation.json"),
    jlens_source_run    = "run_3b_gdn",
)
print(json.dumps(EXP, indent=2))


In [4]:
# Re-resolve dynamically so no stale GPU-box path can survive a copied notebook.
CFG["model_path"] = ai.resolve_ckpt(CFG["ckpt_name"])
assert os.path.basename(CFG["model_path"].rstrip(os.sep)) == CFG["ckpt_name"]


In [5]:
import torch, numpy as np, time
bundle = ai.load_ahn_model(
    CFG["model_path"], dtype=getattr(torch, CFG["dtype"]),
    attn_implementation=CFG["attn_impl"],
    sliding_window=CFG["sliding_window"], num_attn_sinks=CFG["num_attn_sinks"],
)
tok, probe = bundle.tokenizer, ai.AHNProbe(bundle)
if bundle.ahn_impl != CFG["cell"]:
    raise RuntimeError(
        f"loaded AHN cell {bundle.ahn_impl!r}, but {RUN_CONFIG} requires {CFG['cell']!r}"
    )

# Loading the lens and checking Table 3 are two separate questions and used to share a
# try/except. That was a hard blocker: TABLE_3_PASSED is currently False (checks 2 and 3
# fail, see notebook 02), the `except` caught FileNotFoundError only, so the AssertionError
# escaped and killed the notebook here -- before a single measurement -- even though
# README "Next steps" item 4 explicitly says to run this WITH the J-lens.
#
# Production runs require the shared J-lens. A logit-lens fallback is available only
# when REQUIRE_JLENS=False and must remain explicitly exploratory. A missing or failing
# Table 3 stamps lens_validated=False onto every saved row.
lens = None
LENS_VALIDATED = False

if EXP["use_jlens"]:
    try:
        lens = ai.JacobianLens.load(ai.resolve_lens_path(EXP["jlens_path"]),
                                    map_location=str(bundle.model.device))
        print("J-lens loaded, layers:", sorted(lens.jacobians))
    except FileNotFoundError:
        if REQUIRE_JLENS:
            raise
        print("! no J-lens found at", EXP["jlens_path"])
        print("  Falling back to LOGIT LENS. Label every figure 'logit lens, preliminary'.")
        print("  This is not RQ2.")
        EXP["use_jlens"] = False

if EXP["use_jlens"]:
    try:
        with open(EXP["jlens_validation_path"]) as f:
            v = json.load(f)
        LENS_VALIDATED = bool(v.get("TABLE_3_PASSED"))
        print("Table 3 passed:", LENS_VALIDATED)
    except FileNotFoundError:
        print("! shared Table 3 validation not found at", EXP["jlens_validation_path"])
        print("  Restore it beside the shared Qwen2.5-3B J-lens.")
        print("  Proceeding with lens_validated=False.")

    if not LENS_VALIDATED:
        print()
        print("!! PROCEEDING WITH AN UNVALIDATED LENS -- deliberate, not an oversight.")
        print("   Table 3 checks 2 and 3 fail: the J-lens does not reach top-1 on known")
        print("   facts. It does beat the plain logit lens by 8-204x on the same prompts,")
        print("   and RQ2 needs rank SEPARATION between needle and distractor, not top-1.")
        print("   This notebook's control battery (C1/C2/C4) tests that property directly,")
        print("   which is exactly the evidence Gautam asked for before ruling on the lens.")
        print("   Every row is stamped lens_validated=False; label every figure")
        print("   'J-lens, not validated on Table 3' until that ruling lands.")

READOUT = "jlens" if EXP["use_jlens"] else "logit_lens"
if REQUIRE_JLENS and READOUT != "jlens":
    raise RuntimeError("production RULER runs must use readout='jlens'")
if lens is not None:
    missing_layers = sorted(set(EXP["layers"]) - set(lens.jacobians))
    assert not missing_layers, f"shared J-lens is missing layers: {missing_layers}"
EXP["lens_validated"] = LENS_VALIDATED
print()
print("readout:", READOUT, "| lens_validated:", LENS_VALIDATED)


/home/jupyter-dphs-a263/Interpretability-study-of-Artificial-Hippocampus-Networks/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-09-08 20:41:05.811412: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-08 20:41:05.824491: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1788900065.840322 3457312 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered

[2026-09-08 20:41:07,851] [INFO] [real_accelerator.py:222:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/opt/tljh/user/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/opt/tljh/user/compiler_compat/ld: cannot find -lcufile: No such file or directory
collect2: error: ld returned 1 exit status
Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.52it/s]

J-lens loaded, layers: [9, 18, 27]
Table 3 passed: False

!! PROCEEDING WITH AN UNVALIDATED LENS -- deliberate, not an oversight.
   Table 3 checks 2 and 3 fail: the J-lens does not reach top-1 on known
   facts. It does beat the plain logit lens by 8-204x on the same prompts,
   and RQ2 needs rank SEPARATION between needle and distractor, not top-1.
   This notebook's control battery (C1/C2/C4) tests that property directly,
   which is exactly the evidence Gautam asked for before ruling on the lens.
   Every row is stamped lens_validated=False; label every figure
   'J-lens, not validated on Table 3' until that ruling lands.

readout: jlens | lens_validated: False


In [6]:
needles = ai.single_token_needles(tok, EXP["needle_candidates"])
assert len(needles) >= 5, "need at least 5 single-token needles for a usable cohort"

# distractors for control C2: semantically near the needle, absent from the context
DISTRACTORS = {"Paris": "London", "banana": "mango", "Tokyo": "Osaka",
               "violin": "cello", "cinnamon": "nutmeg", "harbour": "wharf",
               "lantern": "torch", "sapphire": "emerald", "meadow": "pasture",
               "trumpet": "clarinet"}
distractor_ids = {}
for n in needles:
    d = DISTRACTORS.get(n)
    ids = tok.encode(f" {d}", add_special_tokens=False) if d else []
    if len(ids) == 1:
        distractor_ids[n] = ids[0]
print(f"{len(distractor_ids)}/{len(needles)} needles have a single-token distractor")


dropped multi-token needles: {'sapphire': 2, 'meadow': 2}
kept 8 single-token needles: ['Paris', 'Tokyo', 'banana', 'cinnamon', 'harbour', 'lantern', 'trumpet', 'violin']
4/8 needles have a single-token distractor


## The measurement

One row per `(needle, eviction distance, filler variant, layer)`. Each row carries
everything Tables 4, 6 and 8 need, plus the four controls, so the whole battery comes
out of one sweep rather than four.


In [7]:
@torch.no_grad()
def measure(needle, needle_id, distance, filler_idx, in_window=False, shuffle=False):
    spec = ai.build_niah_prompt(tok, needle, bundle, eviction_distance=distance,
                                in_window=in_window, filler_idx=filler_idx)
    if not spec["ahn_will_activate"]:
        return []
    if not in_window and not spec["needle_is_evicted"]:
        return []

    prompt = spec["prompt"]
    if shuffle:   # control C3 — destroy word order, keep the token multiset
        w = prompt.split()
        np.random.default_rng(ai.SEED).shuffle(w)
        prompt = " ".join(w)

    ins = tok(prompt, return_tensors="pt").to(bundle.model.device)
    on  = probe.run(ins, nowrite=False, layers=EXP["layers"], capture_residual=True)
    off = probe.run(ins, nowrite=True,  layers=EXP["layers"], capture_residual=True)

    out = []
    for L in EXP["layers"]:
        if L not in on.ahn_raw:
            continue
        o_t = on.o_t(L, pos=-1)
        lg  = ai.readout_logits(o_t, bundle, lens=lens if EXP["use_jlens"] else None, layer=L)

        # C1: residual-stream difference, the non-vacuous zero-state control
        d_res = on.residual(L, pos=-1).float() - off.residual(L, pos=-1).float()
        lg_c1 = ai.readout_logits(d_res, bundle, lens=lens if EXP["use_jlens"] else None, layer=L)

        row = {
            "needle": needle, "layer": L, "readout": READOUT,
            # travels with the data so the caveat cannot be lost between here and
            # a figure caption -- see the lens block above
            "lens_validated": LENS_VALIDATED,
            "requested_distance": distance,
            "eviction_distance": spec["actual_eviction_distance"],
            "in_window": in_window, "shuffled": shuffle, "filler_idx": filler_idx,
            "n_tokens": spec["n_tokens"], "needle_pos": spec["needle_pos"],
            "rank": ai.token_rank(lg, needle_id),
            "p_mem": ai.token_prob(lg, needle_id),
            "entropy": ai.readout_entropy(lg),
            "o_t_norm": float(o_t.float().norm()),
            "rank_c1_residual": ai.token_rank(lg_c1, needle_id),
            "p_mem_c1_residual": ai.token_prob(lg_c1, needle_id),
        }
        if needle in distractor_ids:      # C2
            row["rank_distractor"] = ai.token_rank(lg, distractor_ids[needle])
            row["p_distractor"] = ai.token_prob(lg, distractor_ids[needle])
        out.append(row)
    return out


In [8]:
guard_output("04_retention_rows.json")
rows, t0 = [], time.time()
total = len(needles) * len(EXP["eviction_distances"]) * EXP["n_filler_variants"]
done = 0
for needle, nid in needles.items():
    for dist in EXP["eviction_distances"]:
        for fi in range(EXP["n_filler_variants"]):
            rows += measure(needle, nid, dist, fi)
            done += 1
            if done % 20 == 0:
                print(f"[{done}/{total}] {len(rows)} rows, {(time.time()-t0)/60:.1f} min")
            ai.free_cuda()
print(f"main sweep: {len(rows)} rows in {(time.time()-t0)/60:.1f} min")


[20/168] 60 rows, 1.1 min
[40/168] 120 rows, 2.1 min
[60/168] 180 rows, 3.0 min
[80/168] 240 rows, 3.9 min
[100/168] 300 rows, 4.9 min
[120/168] 360 rows, 5.8 min
[140/168] 420 rows, 6.8 min
[160/168] 480 rows, 7.7 min
main sweep: 504 rows in 8.2 min


In [9]:
# C4 pre-eviction baseline (the ceiling) and C3 shuffled context
ctrl_rows = []
for needle, nid in needles.items():
    for fi in range(EXP["n_filler_variants"]):
        ctrl_rows += measure(needle, nid, 0, fi, in_window=True)          # C4 ceiling
    for dist in (1024, 4096):
        ctrl_rows += measure(needle, nid, dist, 0, shuffle=True)          # C3
    ai.free_cuda()
print(f"control rows: {len(ctrl_rows)}")

all_rows = rows + ctrl_rows
ai.save_json({"rows": all_rows, "cfg": CFG, "exp": EXP,
              "needles": needles, "distractors": distractor_ids},
             "04_retention_rows.json")
print("saved -> 04_retention_rows.json  (this is the file notebook 05 reads)")


control rows: 120
saved -> 04_retention_rows.json  (this is the file notebook 05 reads)


## Table 4 — the control battery, evaluated

C1 and C4 have to pass before Table 6 is filled in. C3 failing is *not* a bug — the
Expected-Tables document flags it as potentially the most publishable result in the
project: if shuffling the context barely changes retention, AHN is closer to a learned
recency mechanism than to content memory, which contradicts the framing of the original
AHN paper.


In [10]:
import numpy as np
V = bundle.vocab
def sel(**kw):
    out = all_rows
    for k, v in kw.items():
        out = [r for r in out if r.get(k) == v]
    return out

main   = [r for r in all_rows if not r["in_window"] and not r["shuffled"]]
inwin  = [r for r in all_rows if r["in_window"]]
shuf   = [r for r in all_rows if r["shuffled"]]

T4 = {}

# C1 — the memory's contribution must beat what the residual difference alone explains,
#      and both must beat chance.
T4["C1_zero_state"] = {
    "mean_rank_o_t": float(np.mean([r["rank"] for r in main])),
    "mean_rank_residual_delta": float(np.mean([r["rank_c1_residual"] for r in main])),
    "chance_rank": V / 2,
    "passed": bool(np.mean([r["rank"] for r in main]) < V / 10),
    "note": "fails if the target is at chance in the memory readout: nothing is retained, "
            "or the readout is still in the wrong basis",
}

# C2 — the true needle must beat a semantically near absent token by >= 1 order of magnitude
withd = [r for r in main if "p_distractor" in r]
if withd:
    ratio = float(np.median([(r["p_mem"] + 1e-12) / (r["p_distractor"] + 1e-12) for r in withd]))
    T4["C2_distractor"] = {"median_prob_ratio": ratio, "n": len(withd),
                           "passed": bool(ratio >= 10.0),
                           "note": "below 10x: the readout reflects topic, not the stored item; "
                                   "RQ2 weakens to 'semantic gist'"}

# C3 — shuffling should hurt retention if the state stores content rather than recency
if shuf:
    T4["C3_shuffled_context"] = {
        "mean_rank_ordered": float(np.mean([r["rank"] for r in main
                                            if r["requested_distance"] in (1024, 4096)])),
        "mean_rank_shuffled": float(np.mean([r["rank"] for r in shuf])),
        "passed": bool(np.mean([r["rank"] for r in shuf])
                       > np.mean([r["rank"] for r in main
                                  if r["requested_distance"] in (1024, 4096)])),
        "note": "FAILURE HERE IS A FINDING, not a bug — see Expected_Tables_and_Figures §3",
    }

# C4 — pre-eviction ceiling must be BETTER than any evicted condition.
#      In the pilot it was worse (Paris baseline rank 110712 vs ~95000 evicted), which
#      is the single clearest sign the measurement was not measuring retention.
if inwin:
    T4["C4_pre_eviction_baseline"] = {
        "mean_rank_in_window": float(np.mean([r["rank"] for r in inwin])),
        "mean_rank_evicted": float(np.mean([r["rank"] for r in main])),
        "passed": bool(np.mean([r["rank"] for r in inwin])
                       < np.mean([r["rank"] for r in main])),
        "note": "if the in-window ceiling is worse than the evicted condition, the "
                "placement or the readout is wrong. Stop and fix before Table 6.",
    }

T4["BATTERY_PASSED"] = bool(T4["C1_zero_state"]["passed"]
                            and T4.get("C4_pre_eviction_baseline", {}).get("passed", True))
ai.save_json(T4, "04_table4_controls.json")
print(json.dumps(T4, indent=2))
print("\nC1+C4:", "PASS -> Table 6 may be populated" if T4["BATTERY_PASSED"]
      else "FAIL -> fix instrumentation, do NOT report Table 6")


{
  "C1_zero_state": {
    "mean_rank_o_t": 87555.69444444444,
    "mean_rank_residual_delta": 100136.74404761905,
    "chance_rank": 75968.0,
    "passed": false,
    "note": "fails if the target is at chance in the memory readout: nothing is retained, or the readout is still in the wrong basis"
  },
  "C2_distractor": {
    "median_prob_ratio": 1.0000000000002065,
    "n": 252,
    "passed": false,
    "note": "below 10x: the readout reflects topic, not the stored item; RQ2 weakens to 'semantic gist'"
  },
  "C3_shuffled_context": {
    "mean_rank_ordered": 88507.05555555556,
    "mean_rank_shuffled": 72947.52083333333,
    "passed": false,
    "note": "FAILURE HERE IS A FINDING, not a bug \u2014 see Expected_Tables_and_Figures \u00a73"
  },
  "C4_pre_eviction_baseline": {
    "mean_rank_in_window": 77489.48611111111,
    "mean_rank_evicted": 87555.69444444444,
    "passed": true,
    "note": "if the in-window ceiling is worse than the evicted condition, the placement or the readout 

## Adding more metrics

In [11]:
import json, numpy as np
data = json.load(open(os.path.join(CFG["results_dir"], "04_retention_rows.json")))
rows = data["rows"]
main = [r for r in rows if not r["in_window"] and not r["shuffled"]]

for L in sorted({r["layer"] for r in main}):
    rs = [r for r in main if r["layer"] == L]
    print(f"layer {L}: mean_rank={np.mean([r['rank'] for r in rs]):.0f}  "
          f"mean_p_mem={np.mean([r['p_mem'] for r in rs]):.3e}")

# raw p_mem / p_distractor pairs, unrounded, no epsilon
withd = [r for r in main if "p_distractor" in r][:10]
for r in withd:
    print(r["needle"], r["layer"], r["eviction_distance"],
          "p_mem=", r["p_mem"], "p_distractor=", r["p_distractor"])

layer 9: mean_rank=90789  mean_p_mem=4.172e-19
layer 18: mean_rank=95713  mean_p_mem=2.608e-07
layer 27: mean_rank=76165  mean_p_mem=6.413e-07
Paris 9 80 p_mem= 8.447425495694199e-22 p_distractor= 1.8888208845315228e-21
Paris 18 80 p_mem= 2.4926640307398884e-08 p_distractor= 9.295327174640988e-09
Paris 27 80 p_mem= 1.0627352367009735e-06 p_distractor= 1.6349940779036842e-07
Paris 9 87 p_mem= 1.9742712590594074e-18 p_distractor= 7.004958503326124e-17
Paris 18 87 p_mem= 5.521575210942764e-11 p_distractor= 8.433170134436452e-11
Paris 27 87 p_mem= 1.9520651761695262e-08 p_distractor= 3.5069367410045515e-09
Paris 9 84 p_mem= 1.7581360665144967e-22 p_distractor= 2.6905215877197896e-20
Paris 18 84 p_mem= 5.978807848805445e-07 p_distractor= 3.008181792552023e-08
Paris 27 84 p_mem= 3.1332701837527566e-06 p_distractor= 5.043765440859715e-07
Paris 9 272 p_mem= 4.111064593872897e-24 p_distractor= 9.194856215033909e-24


In [12]:
import json, numpy as np

data = json.load(open(os.path.join(CFG["results_dir"], "04_retention_rows.json")))
rows = data["rows"]
main = [r for r in rows if not r["in_window"] and not r["shuffled"]]
V = 151936
EPS = 1e-30  # small enough not to swamp probabilities down to ~1e-24

print("=== C1 per layer (pass bar: mean_rank < %d) ===" % (V // 10))
for L in sorted({r["layer"] for r in main}):
    rs = [r for r in main if r["layer"] == L]
    mr = np.mean([r["rank"] for r in rs])
    print(f"layer {L}: n={len(rs)} mean_rank={mr:.0f}  passes={mr < V/10}")

print("\n=== C2 per layer, corrected epsilon (pass bar: median ratio >= 10) ===")
for L in sorted({r["layer"] for r in main}):
    rs = [r for r in main if r["layer"] == L and "p_distractor" in r]
    ratios = [(r["p_mem"] + EPS) / (r["p_distractor"] + EPS) for r in rs]
    print(f"layer {L}: n={len(rs)}  median_ratio={np.median(ratios):.3f}  "
          f"frac_needle>distractor={np.mean([r>1 for r in ratios]):.2f}  "
          f"frac_pass_10x={np.mean([r>=10 for r in ratios]):.2f}")

=== C1 per layer (pass bar: mean_rank < 15193) ===
layer 9: n=168 mean_rank=90789  passes=False
layer 18: n=168 mean_rank=95713  passes=False
layer 27: n=168 mean_rank=76165  passes=False

=== C2 per layer, corrected epsilon (pass bar: median ratio >= 10) ===
layer 9: n=84  median_ratio=0.695  frac_needle>distractor=0.46  frac_pass_10x=0.25
layer 18: n=84  median_ratio=0.550  frac_needle>distractor=0.38  frac_pass_10x=0.10
layer 27: n=84  median_ratio=4.911  frac_needle>distractor=0.75  frac_pass_10x=0.17


### Gate for this notebook

- [ ] C1 passes (target well inside the top decile of vocabulary, not at chance)
- [ ] C4 passes (in-window ceiling beats every evicted condition)
- [ ] C2 recorded — if the ratio is under 10×, RQ2's claim weakens to "semantic gist"
- [ ] C3 recorded — **if it fails, message Gautam before doing anything else**
- [ ] `04_retention_rows.json` saved

Analysis and figures are in **05_analysis_and_figures.ipynb**, which runs on CPU. Download
`04_retention_rows.json` and run 05 on your laptop — GPU time is the scarce resource,
analysis time is not.


## RULER NIAH cohort — the primary RQ2 cohort, wired up 7 Sep

Expected Tables row 1 of Table 2 makes RULER NIAH n=60 the PRIMARY RQ2 cohort — "ground
truth y* is known by construction" is the proposal's number-one reason-to-believe. Every
NIAH result above uses the homemade `build_niah_prompt` instead: 8 single-token needles x
3 filler variants x 7 distances. `ai.load_ruler()` has existed since this notebook's first
sweep and was never called.

This is the cheapest direct test of candidate (b) from Next steps item 5:
`build_niah_prompt` ends at `"What was the special word?"` with no answer prefix;
`load_ruler` appends RULER's own `answer_prefix`. The readout stays at `pos=-1` in both, so
prompt construction is the only variable that changes between this sweep and the one above.

**What this is not:** a replacement for C1–C4. RULER gives one prompt and one gold answer
per example, not a needle/distractor pair or a way to shuffle just the needle region, so
there is no C2, C3 or C4 analog here — only a C1-shaped question (does the readout find the
real answer above chance in a real cohort). That is exactly the question that matters for
candidate (b), and nothing more should be claimed for it.

In [6]:
# config="8192" looked right by construction (sliding_window + num_attn_sinks = 8192)
# but was WRONG in practice: measured on this box, 7 Sep --
#   config=4096   n_tokens min=3675  median=3681  max=3687
#   config=8192   n_tokens min=7875  median=7881  max=7887   <- under 8192, 60/60 skipped
#   config=16384  n_tokens min=15678 median=15679 max=15684
# simonjegou/ruler's length buckets were built with a different tokenizer than Qwen's;
# Qwen's larger vocab (151,936) compresses English text into noticeably fewer tokens, so
# a bucket built to be "8192 tokens" comes out under 8192 once retokenized here. Only
# 4096/8192/16384 exist as configs for this dataset -- there is no 32768 to fall back to.
# 16384 is the only bucket that clears window+sinks, with ~7,500 tokens of margin and a
# 6-token spread across examples, so it should activate for very close to all 60.
RULER_CFG = dict(config="16384", split="test", n=60, seed=ai.SEED)
ruler_examples = ai.load_ruler(**RULER_CFG)
print(f"loaded {len(ruler_examples)} RULER NIAH examples, config={RULER_CFG['config']}")
print("--- tail of example 0, sanity check the prompt ends where expected ---")
print(ruler_examples[0]["prompt"][-300:])
print("gold answer:", ruler_examples[0]["answer"])

loaded 60 RULER NIAH examples, config=16384
--- tail of example 0, sanity check the prompt ends where expected ---
blue. The sun is yellow. Here we go. There and back again.
The grass is green. The sky is blue. The sun is yellow. Here we go. There and back again.
What is the special magic number for solid-few mentioned in the provided text? The special magic number for solid-few mentioned in the provided text is
gold answer: 7700828


In [14]:
@torch.no_grad()
def measure_ruler(example):
    prompt, answer = example["prompt"], example["answer"]
    # Gold answer's FIRST token only, same convention the RQ3 join already uses for the
    # same reason: the readout is one next-token distribution, not a generation. The
    # leading space matters for Qwen's BPE -- most words tokenize differently with vs.
    # without it, and answer_prefix already ends without a trailing space.
    target_id = tok.encode(" " + answer.lstrip(), add_special_tokens=False)[0]

    ins = tok(prompt, return_tensors="pt").to(bundle.model.device)
    n_tokens = int(ins["input_ids"].shape[1])
    window = bundle.sliding_window or 0
    sinks = bundle.num_attn_sinks or 0
    if not (n_tokens > window + sinks):
        return None   # AHN never activates at this length -- same check build_niah_prompt uses

    hit = prompt.find(answer)
    if hit < 0:
        raise ValueError("RULER needle answer was not found in its prompt")
    needle_pos = len(tok(prompt[:hit], add_special_tokens=True)["input_ids"])
    window_start = n_tokens - window
    eviction_distance = window_start - needle_pos
    needle_is_evicted = sinks <= needle_pos < window_start

    # 16384-config prompts run ~1.7x longer than anything else in this notebook, on a
    # 20 GB MIG slice shared with 3 other people (already OOM'd once today on a slice
    # someone else was using). A transient OOM here should not lose the whole 60-example
    # sweep -- skip the example, log why, and keep going.
    try:
        on  = probe.run(ins, nowrite=False, layers=EXP["layers"], capture_residual=True)
        off = probe.run(ins, nowrite=True,  layers=EXP["layers"], capture_residual=True)
    except torch.cuda.OutOfMemoryError as e:
        ai.free_cuda()
        print(f"  ! OOM at n_tokens={n_tokens}, skipping this example: {e}")
        return None

    out = []
    for L in EXP["layers"]:
        if L not in on.ahn_raw:
            continue
        o_t = on.o_t(L, pos=-1)
        lg  = ai.readout_logits(o_t, bundle, lens=lens if EXP["use_jlens"] else None, layer=L)

        d_res = on.residual(L, pos=-1).float() - off.residual(L, pos=-1).float()
        lg_c1 = ai.readout_logits(d_res, bundle, lens=lens if EXP["use_jlens"] else None, layer=L)

        out.append({
            "task": example["task"], "layer": L, "readout": READOUT,
            "lens_validated": LENS_VALIDATED,
            "cohort": "ruler_niah", "ruler_config": RULER_CFG["config"],
            "n_tokens": n_tokens,
            "needle_pos": needle_pos,
            "compression_boundary": window_start,
            "eviction_distance": eviction_distance,
            "needle_is_evicted": needle_is_evicted,
            "rank": ai.token_rank(lg, target_id),
            "p_mem": ai.token_prob(lg, target_id),
            "entropy": ai.readout_entropy(lg),
            "rank_c1_residual": ai.token_rank(lg_c1, target_id),
            "p_mem_c1_residual": ai.token_prob(lg_c1, target_id),
        })
    return out

In [15]:
guard_output("04g_ruler_niah_rows.json")
ruler_rows, skipped, t0 = [], 0, time.time()
for i, ex in enumerate(ruler_examples):
    res = measure_ruler(ex)
    if res is None:
        skipped += 1
    else:
        ruler_rows += res
    if (i + 1) % 10 == 0:
        print(f"[{i+1}/{len(ruler_examples)}] {len(ruler_rows)} rows, {skipped} skipped "
              f"(too short for AHN to activate), {(time.time()-t0)/60:.1f} min")
    ai.free_cuda()

print(f"RULER sweep: {len(ruler_rows)} rows from {len(ruler_examples) - skipped}/"
      f"{len(ruler_examples)} examples in {(time.time()-t0)/60:.1f} min")

if skipped > len(ruler_examples) // 2:
    print("! more than half the cohort skipped -- selected RULER config is landing short of "
          "window+sinks for most examples. Recheck RULER_CFG before trusting the rows below.")

ai.save_json({"rows": ruler_rows, "cfg": CFG, "exp": EXP, "ruler_cfg": RULER_CFG},
             "04g_ruler_niah_rows.json")
print("saved -> 04g_ruler_niah_rows.json")

[10/60] 30 rows, 0 skipped (too short for AHN to activate), 0.6 min
[20/60] 60 rows, 0 skipped (too short for AHN to activate), 1.2 min
[30/60] 90 rows, 0 skipped (too short for AHN to activate), 1.7 min
[40/60] 120 rows, 0 skipped (too short for AHN to activate), 2.3 min
[50/60] 150 rows, 0 skipped (too short for AHN to activate), 2.9 min
[60/60] 180 rows, 0 skipped (too short for AHN to activate), 3.5 min
RULER sweep: 180 rows from 60/60 examples in 3.5 min
saved -> 04g_ruler_niah_rows.json


### Gate for this section

- [ ] `skipped` is a small fraction of 60 -- most examples actually reach AHN activation
      at config="16384"
- [ ] median `rank` well below chance (~76,000) would be the first real-cohort signal in
      candidate (b)'s favour; at or above chance keeps (b) alive as the explanation
- [ ] compare against `04_retention_rows.json` rows at a matched eviction distance before
      drawing any conclusion -- one cohort's median against the other's is the whole point

## RULER control battery — REBUILT 7 Sep after a target bug

**The first version of this section, and run 025 itself, scored the wrong token.**
RULER NIAH answers here are 7-digit numbers, and Qwen tokenizes `" 7700828"` as
`[' ', '7', '7', '0', '0', '8', '2', '8']` — the leading space is its own token. So
`encode(" " + answer)[0]` returned token 220, a bare space, **identically for all 60
examples**. Run 025 measured the rank of a space character, not of the needle. Its C1
numbers do not mean what the 7 Sep findings entry says they mean, and C2 could not run at
all: with one distinct target token every pair dropped as a collision.

**The fix: score the answer as a sequence.** Append the gold answer to the prompt, run
once, and read out at the seven positions that predict the seven digits. At each position
record the log-probability of all ten digit tokens. From that 7x10 table per example:

- **C1** — the log-probability of the example's *own* answer, and the rank of the correct
  digit at each position.
- **C2** — the log-probability of any *other* example's answer, at the same positions,
  from the same table. The full 60x60 cross matrix falls out with no extra compute, and
  the baseline correction is the same ratio-of-ratios as before, now over answer
  log-probabilities instead of single-token probabilities.

Digits are shared across all answers, so pair-identity baseline correction is not
optional here — it is the whole measurement. That is exactly what C2 is for.

C3-context and C3-lens are unchanged in design, rescored against the new target.

In [8]:
import collections

In [9]:
# --- targets: digit tokens, and each example's answer as a digit sequence -----------
DIGIT_IDS = [tok.encode(str(d), add_special_tokens=False)[0] for d in range(10)]
assert len(set(DIGIT_IDS)) == 10, "digits are not single distinct tokens"
print("digit token ids 0-9:", DIGIT_IDS)

def answer_digits(ans):
    """Gold answer as a list of digit values. None if it is not all digits."""
    a = ans.strip()
    return [int(ch) for ch in a] if a.isdigit() else None

answer_digit_seqs = [answer_digits(ex["answer"]) for ex in ruler_examples]
bad = [i for i, d in enumerate(answer_digit_seqs) if d is None]
lens_seen = collections.Counter(len(d) for d in answer_digit_seqs if d)
print(f"answers parsed as digits: {len(answer_digit_seqs) - len(bad)}/{len(ruler_examples)}",
      f"| lengths: {dict(lens_seen)}")
if bad:
    print("  ! non-numeric answers at", bad[:10], "-- these are skipped")
ANSWER_LEN = lens_seen.most_common(1)[0][0]
print(f"scoring the first {ANSWER_LEN} digits of each answer")

lens_shuf = lens.shuffled(seed=ai.SEED) if lens is not None else None
print("shuffled lens ready:", lens_shuf is not None)
lens_perm_layers = lens.permuted_layers() if lens is not None else None
print("layer-permuted lens ready:", lens_perm_layers is not None)
WINDOW, SINKS = bundle.sliding_window or 0, bundle.num_attn_sinks or 0

digit token ids 0-9: [15, 16, 17, 18, 19, 20, 21, 22, 23, 24]
answers parsed as digits: 60/60 | lengths: {7: 60}
scoring the first 7 digits of each answer
shuffled lens ready: True


In [10]:
@torch.no_grad()
def measure_ruler_seq(idx, example):
    """Score the answer digit by digit, and record the full digit distribution.

    The readout at position p predicts the token at p+1. Appending the gold answer to the
    prompt gives ANSWER_LEN positions whose correct next token is a known digit, so one
    forward-pass pair yields the whole 7x10 log-probability table -- which is enough to
    score this example's answer (C1) and every other example's answer (C2) offline.
    """
    digits = answer_digit_seqs[idx]
    if digits is None:
        return []
    digits = digits[:ANSWER_LEN]

    out = []
    for condition in ("ordered", "shuffled_context"):
        if condition == "ordered":
            base = example["prompt"]
        else:
            w = example["context"].split()
            np.random.default_rng(ai.SEED + idx).shuffle(w)
            base = " ".join(w) + example["question"] + example["answer_prefix"]

        prompt_ids = tok(base, return_tensors="pt")["input_ids"]
        full = base + " " + "".join(str(d) for d in digits)
        ins = tok(full, return_tensors="pt").to(bundle.model.device)
        n_tokens = int(ins["input_ids"].shape[1])
        P = int(prompt_ids.shape[1])

        # BPE can merge across the join; if the prompt is not a clean prefix the position
        # arithmetic below is wrong, so skip rather than silently mis-score.
        if not torch.equal(ins["input_ids"][0, :P].cpu(), prompt_ids[0]):
            print(f"  ! example {idx} ({condition}): prompt is not a token prefix of "
                  f"prompt+answer, skipped")
            return []
        # tokens P..P+len(digits) are [space, d0, d1, ...]; digit t is predicted at P+t
        if n_tokens < P + 1 + len(digits):
            print(f"  ! example {idx} ({condition}): answer did not tokenize to "
                  f"{len(digits)} digits, skipped")
            return []
        if not (n_tokens > WINDOW + SINKS):
            return []

        hit = base.find(example["answer"])
        window_start = n_tokens - WINDOW
        if hit < 0:
            print(f"  ! example {idx} ({condition}): needle answer not found, skipped")
            return []
        needle_pos = len(tok(base[:hit], add_special_tokens=True)["input_ids"])
        eviction_distance = window_start - needle_pos
        needle_is_evicted = SINKS <= needle_pos < window_start
        placement = ("in_sink_region" if needle_pos < SINKS else
                     "evicted" if needle_is_evicted else "in_window")

        try:
            on  = probe.run(ins, nowrite=False, layers=EXP["layers"], capture_residual=True)
            off = probe.run(ins, nowrite=True,  layers=EXP["layers"], capture_residual=True)
        except torch.cuda.OutOfMemoryError as e:
            ai.free_cuda()
            print(f"  ! OOM on example {idx} ({condition}) at n_tokens={n_tokens}: {e}")
            return []

        for L in EXP["layers"]:
            if L not in on.ahn_raw:
                continue
            tbl, tbl_shuf, tbl_perm, ranks = [], [], [], []
            for t in range(len(digits)):
                pos = P + t          # predicts the token at P+t+1, i.e. digit t
                d_res = (on.residual(L, pos=pos).float() - off.residual(L, pos=pos).float())
                lg = ai.readout_logits(d_res, bundle, lens=lens, layer=L)
                lp = torch.log_softmax(lg.float(), dim=-1)
                tbl.append([float(lp[i]) for i in DIGIT_IDS])
                ranks.append(ai.token_rank(lg, DIGIT_IDS[digits[t]]))
                if lens_shuf is not None:
                    lg_s = ai.readout_logits(d_res, bundle, lens=lens_shuf, layer=L)
                    lp_s = torch.log_softmax(lg_s.float(), dim=-1)
                    tbl_shuf.append([float(lp_s[i]) for i in DIGIT_IDS])
                if lens_perm_layers is not None:
                    lg_p = ai.readout_logits(d_res, bundle, lens=lens_perm_layers, layer=L)
                    lp_p = torch.log_softmax(lg_p.float(), dim=-1)
                    tbl_perm.append([float(lp_p[i]) for i in DIGIT_IDS])

            row = {
                "example": idx, "layer": L, "condition": condition,
                "readout": READOUT, "lens_validated": LENS_VALIDATED,
                "cohort": "ruler_niah", "ruler_config": RULER_CFG["config"],
                "n_tokens": n_tokens, "needle_pos": needle_pos,
                "compression_boundary": window_start,
                "eviction_distance": eviction_distance,
                "needle_is_evicted": needle_is_evicted,
                "placement": placement,
                "answer_digits": digits,
                # [position][digit] log-probability -- everything C1 and C2 need
                "digit_logprobs": tbl,
                "digit_ranks": ranks,
                "answer_logprob": float(sum(tbl[t][digits[t]] for t in range(len(digits)))),
                "mean_digit_rank": float(sum(ranks) / len(ranks)),
            }
            if tbl_shuf:
                row["digit_logprobs_shuffled_lens"] = tbl_shuf
                row["answer_logprob_shuffled_lens"] = float(
                    sum(tbl_shuf[t][digits[t]] for t in range(len(digits))))
            if tbl_perm:
                row["digit_logprobs_permuted_layers"] = tbl_perm
                row["answer_logprob_permuted_layers"] = float(
                    sum(tbl_perm[t][digits[t]] for t in range(len(digits))))
            out.append(row)
    return out

In [11]:
guard_output("04i_ruler_controls_rows.json")
ctrl_rows, t0 = [], time.time()
for idx, ex in enumerate(ruler_examples):
    ctrl_rows += measure_ruler_seq(idx, ex)
    if (idx + 1) % 10 == 0:
        print(f"[{idx+1}/{len(ruler_examples)}] {len(ctrl_rows)} rows, "
              f"{(time.time()-t0)/60:.1f} min")
    ai.free_cuda()

print(f"control battery: {len(ctrl_rows)} rows in {(time.time()-t0)/60:.1f} min")
print("placement (ordered):",
      dict(collections.Counter(r["placement"] for r in ctrl_rows if r["condition"] == "ordered")))

ai.save_json({"rows": ctrl_rows, "cfg": CFG, "exp": EXP, "ruler_cfg": RULER_CFG,
              "digit_ids": DIGIT_IDS, "answer_len": ANSWER_LEN,
              "answer_digit_seqs": answer_digit_seqs},
             "04i_ruler_controls_rows.json")
print("saved -> 04i_ruler_controls_rows.json")
print(f"analyse with: python ruler_controls.py --run-config {CFG['run_name']}")

[10/60] 60 rows, 1.6 min
[20/60] 120 rows, 2.8 min
[30/60] 180 rows, 4.0 min
[40/60] 240 rows, 5.1 min
[50/60] 300 rows, 6.3 min
[60/60] 360 rows, 7.5 min
control battery: 360 rows in 7.5 min
placement (ordered): {'in_window': 84, 'evicted': 96}
saved -> 04i_ruler_controls_rows.json   (analyse with: python ruler_controls.py)


### Gate for the RULER control battery

- [ ] **C3-lens first.** If the signal survives a row-permuted map, everything below is a
      decoding artefact and the rest of the battery is moot
- [ ] C1 restated on the sequence target: mean digit rank against chance, and own-answer
      log-probability against the cohort's other answers. **Run 025's rank-17,250 figure
      does not carry over** — it was the space token
- [ ] C2 per-example effect with a CI excluding 1.0; the pre-registered bar is 10x, and
      anything between is reported as real-but-below-bar, not rounded up
- [ ] C3-context order sensitivity, direction stated as measured
- [ ] Everything restricted to `placement == "evicted"`

## Cross-check: does Sơn's layer-27 C2 result survive a permuted lens?

Sơn's multi-control C2 statistic (`04-C2-debug.ipynb`, cells 105-110) on the homemade
4-needle set finds layer 27 **significantly negative** (0.863x [0.768, 0.969], p=0.016) --
the opposite sign from RULER's layer 27 (1.672x [1.446, 1.929], p<0.0001, verified real via
C3-lens). Same layer, same checkpoint, opposite direction, neither has been checked against
a permuted lens on his design.

This replicates his exact sweep (same 4 targets, same 4 controls, same distances/fillers --
168 forward passes total) and decodes every readout through BOTH the real J-lens and a
row-permuted one, the same C3-lens check `ruler_controls.py` already applies to RULER. His
own notebook never cached the raw `o_t` vectors, only the decoded probabilities, so this
reruns the forward passes rather than re-reading anything -- deterministic given the same
config, so it reproduces his numbers as a sanity check before answering the new question.

**Reading the result:** if layer 27's negative effect collapses under the permuted lens
(CI moves toward 1, or the direction becomes unstable), it's a decoding artifact and
RULER's positive result stands uncontested. If it survives -- stays negative and
significant under permutation -- both results are independently verified real, pointing in
opposite directions, and that is a bigger and more interesting problem than either result
alone.

In [6]:
# Exact replication of 04-C2-debug.ipynb's needle/control/distance/filler grid.
# Independent of that notebook's variables -- self-contained here.
import numpy as np, pandas as pd, time
from scipy import stats as _stats

TARGETS  = ["Paris", "Tokyo", "banana", "lantern"]
CONTROLS = ["river", "chair", "window", "garden"]
LAYERS   = [9, 18, 27]
DISTANCES = EXP["eviction_distances"]
FILLERS   = range(EXP["n_filler_variants"])

target_ids = {t: tok.encode(f" {t}", add_special_tokens=False)[0] for t in TARGETS}
for t, tid in target_ids.items():
    assert len(tok.encode(f" {t}", add_special_tokens=False)) == 1, f"{t} is not single-token"

lens_shuf = lens.shuffled(seed=ai.SEED) if lens is not None else None
print("real lens:", lens is not None, "| shuffled lens:", lens_shuf is not None)
print(f"grid: {len(TARGETS)} targets + {len(CONTROLS)} controls x "
      f"{len(list(DISTANCES))} distances x {len(list(FILLERS))} fillers "
      f"= {(len(TARGETS)+len(CONTROLS))*len(list(DISTANCES))*len(list(FILLERS))} forward passes")

real lens: True | shuffled lens: True
grid: 4 targets + 4 controls x 7 distances x 3 fillers = 168 forward passes


In [7]:
@torch.no_grad()
def score_stored_word(stored_word, distance, filler_idx):
    """One forward pass -> p(target) for every target, under BOTH lenses.

    Mirrors 04-C2-debug.ipynb cells 40 and 107: when stored_word is itself a TARGET this
    gives the "original" condition (target stored, tested with itself); when stored_word
    is a CONTROL this gives the multi-control baseline (unrelated word stored, every
    target still tested).
    """
    spec = ai.build_niah_prompt(tok, stored_word, bundle, eviction_distance=distance,
                                 in_window=False, filler_idx=filler_idx)
    if not spec["needle_is_evicted"]:
        return []
    ins = tok(spec["prompt"], return_tensors="pt").to(bundle.model.device)
    on = probe.run(ins, nowrite=False, layers=LAYERS, capture_residual=False)

    out = []
    for L in LAYERS:
        o_t = on.o_t(L, pos=-1)
        lg      = ai.readout_logits(o_t, bundle, lens=lens,      layer=L)
        lg_shuf = ai.readout_logits(o_t, bundle, lens=lens_shuf, layer=L) if lens_shuf is not None else None
        for target in TARGETS:
            row = {
                "stored_word": stored_word, "is_target": stored_word in TARGETS,
                "tested_target": target, "layer": L,
                "requested_distance": distance, "filler_idx": filler_idx,
                "p_target_real": ai.token_prob(lg, target_ids[target]),
            }
            if lg_shuf is not None:
                row["p_target_shuffled"] = ai.token_prob(lg_shuf, target_ids[target])
            out.append(row)
    return out

rows, t0 = [], time.time()
words = TARGETS + CONTROLS
total = len(words) * len(list(DISTANCES)) * len(list(FILLERS))
done = 0
for word in words:
    for distance in DISTANCES:
        for filler_idx in FILLERS:
            rows += score_stored_word(word, distance, filler_idx)
            done += 1
            if done % 20 == 0:
                print(f"[{done}/{total}] {len(rows)} rows, {(time.time()-t0)/60:.1f} min")
            ai.free_cuda()
print(f"done: {len(rows)} rows in {(time.time()-t0)/60:.1f} min")

df_check = pd.DataFrame(rows)
ai.save_json({"rows": rows, "cfg": CFG, "targets": TARGETS, "controls": CONTROLS},
             "04j_c2_multicontrol_lens_check.json")
print("saved -> 04j_c2_multicontrol_lens_check.json")

[20/168] 240 rows, 0.8 min
[40/168] 480 rows, 1.3 min
[60/168] 720 rows, 1.9 min
[80/168] 960 rows, 2.4 min
[100/168] 1200 rows, 2.9 min
[120/168] 1440 rows, 3.5 min
[140/168] 1680 rows, 4.0 min
[160/168] 1920 rows, 4.6 min
done: 2016 rows in 4.8 min
saved -> 04j_c2_multicontrol_lens_check.json


In [8]:
def multicontrol_stat(df, prob_col):
    """Reproduces 04-C2-debug.ipynb cells 108/110's statistic for one probability column."""
    orig = df[(df["is_target"]) & (df["stored_word"] == df["tested_target"])][
        ["tested_target", "layer", "requested_distance", "filler_idx", prob_col]
    ].rename(columns={"tested_target": "target", prob_col: "p_target_original"})

    ctrl = df[~df["is_target"]].copy()
    ctrl["log_p"] = np.log(ctrl[prob_col] + 1e-30)
    ctrl_mean = (ctrl.groupby(["tested_target", "layer", "requested_distance", "filler_idx"])
                     ["log_p"].mean().reset_index(name="mean_log_p_control")
                     .rename(columns={"tested_target": "target"}))

    merged = orig.merge(ctrl_mean, on=["target", "layer", "requested_distance", "filler_idx"])
    merged["delta_log"] = np.log(merged["p_target_original"] + 1e-30) - merged["mean_log_p_control"]

    cond = merged.groupby(["layer", "requested_distance", "filler_idx"])["delta_log"].mean().reset_index()

    out = []
    for L in sorted(cond["layer"].unique()):
        x = cond.loc[cond["layer"] == L, "delta_log"].to_numpy()
        n = len(x); mean_log = x.mean(); se = x.std(ddof=1) / np.sqrt(n)
        tcrit = _stats.t.ppf(0.975, df=n - 1)
        t_stat, p = _stats.ttest_1samp(x, popmean=0.0)
        out.append({"layer": int(L), "n": n, "fold": float(np.exp(mean_log)),
                    "ci_low": float(np.exp(mean_log - tcrit * se)),
                    "ci_high": float(np.exp(mean_log + tcrit * se)),
                    "p_value": float(p)})
    return out

real = multicontrol_stat(df_check, "p_target_real")
shuf = multicontrol_stat(df_check, "p_target_shuffled") if "p_target_shuffled" in df_check else None

print(f"{'layer':>5} {'REAL lens':>28} {'SHUFFLED lens':>28} {'verdict':>16}")
comparison = []
for r in real:
    s = next((x for x in shuf if x["layer"] == r["layer"]), None) if shuf else None
    real_sig = not (r["ci_low"] < 1.0 < r["ci_high"])
    if s is None:
        verdict = "no shuffled lens"
    else:
        shuf_sig = not (s["ci_low"] < 1.0 < s["ci_high"])
        verdict = ("COLLAPSES (artifact)" if real_sig and not shuf_sig else
                    "SURVIVES (real, disagrees w/ RULER)" if real_sig and shuf_sig else
                    "not significant either way")
    print(f"{r['layer']:>5} "
          f"{r['fold']:>7.3f}x [{r['ci_low']:.3f},{r['ci_high']:.3f}] p={r['p_value']:.4f}  "
          f"{(s['fold'] if s else float('nan')):>7.3f}x "
          f"[{(s['ci_low'] if s else float('nan')):.3f},{(s['ci_high'] if s else float('nan')):.3f}] "
          f"p={(s['p_value'] if s else float('nan')):.4f}   {verdict:>16}")
    comparison.append({"layer": r["layer"], "real": r, "shuffled": s, "verdict": verdict})

ai.save_json({"comparison": comparison}, "04j_c2_multicontrol_lens_stats.json")
print("saved -> 04j_c2_multicontrol_lens_stats.json")

layer                    REAL lens                SHUFFLED lens          verdict
    9   0.930x [0.868,0.997] p=0.0417    1.000x [0.991,1.010] p=0.9321   COLLAPSES (artifact)
   18   1.069x [0.947,1.207] p=0.2616    0.982x [0.897,1.075] p=0.6788   not significant either way
   27   0.873x [0.779,0.978] p=0.0218    1.118x [1.070,1.168] p=0.0000   SURVIVES (real, disagrees w/ RULER)
saved -> 04j_c2_multicontrol_lens_stats.json


### Reading this result

- **Layer 27 COLLAPSES under the shuffled lens** -> Sơn's negative result was a decoding
  artifact. RULER's verified positive result stands as the only real evidence at layer 27.
- **Layer 27 SURVIVES** -> two independently-verified-real effects, opposite signs, same
  layer, same checkpoint. This is a bigger finding than either result alone and needs its
  own line in the diagnosis packet -- not a tiebreak, a genuine open question about
  content-dependence (common words / place names vs. digit sequences).

## Proper permutation test: 20 shuffled lenses, not one

The single-shuffle check above was inconclusive in a specific way: layer 27's shuffled-lens
result (1.118x, p<0.0001) was significant, tighter, and **opposite in sign** to the real
result (0.873x) -- not "reverted toward chance" the way a genuine artifact-check should
look. With one arbitrary permutation there is no way to tell a real structural artifact
from an unlucky draw. This builds an actual null distribution: 20 independently-seeded
shuffled lenses, and asks where the real-lens statistic falls relative to them.

Reuses the same 168-forward-pass grid -- the extra cost is 20 more matrix multiplies per
already-computed `o_t` (the readout decode), not 20 more forward passes through the model,
so this stays cheap.

In [9]:
# 20 independently-seeded shuffled lenses, distinct from the single check above (which
# used ai.SEED). Distinct seeds so this is a real sample from the permutation distribution,
# not 20 copies of a similar draw.
PERM_SEEDS = list(range(1, 21))
lens_perms = [lens.shuffled(seed=s) for s in PERM_SEEDS] if lens is not None else []
print(f"built {len(lens_perms)} independently-seeded shuffled lenses")

built 20 independently-seeded shuffled lenses


In [10]:
@torch.no_grad()
def score_stored_word_multiperm(stored_word, distance, filler_idx):
    """Same grid as the single-shuffle check, but decodes each o_t through the real lens
    AND all 20 shuffled lenses. Long format: one row per (lens_id, tested_target)."""
    spec = ai.build_niah_prompt(tok, stored_word, bundle, eviction_distance=distance,
                                 in_window=False, filler_idx=filler_idx)
    if not spec["needle_is_evicted"]:
        return []
    ins = tok(spec["prompt"], return_tensors="pt").to(bundle.model.device)
    on = probe.run(ins, nowrite=False, layers=LAYERS, capture_residual=False)

    out = []
    for L in LAYERS:
        o_t = on.o_t(L, pos=-1)
        readouts = [("real", ai.readout_logits(o_t, bundle, lens=lens, layer=L))]
        readouts += [(f"perm_{i:02d}", ai.readout_logits(o_t, bundle, lens=lp, layer=L))
                     for i, lp in enumerate(lens_perms)]
        for lens_id, lg in readouts:
            for target in TARGETS:
                out.append({
                    "stored_word": stored_word, "is_target": stored_word in TARGETS,
                    "tested_target": target, "layer": L,
                    "requested_distance": distance, "filler_idx": filler_idx,
                    "lens_id": lens_id, "p_target": ai.token_prob(lg, target_ids[target]),
                })
    return out

rows_mp, t0 = [], time.time()
words = TARGETS + CONTROLS
total = len(words) * len(list(DISTANCES)) * len(list(FILLERS))
done = 0
for word in words:
    for distance in DISTANCES:
        for filler_idx in FILLERS:
            rows_mp += score_stored_word_multiperm(word, distance, filler_idx)
            done += 1
            if done % 20 == 0:
                print(f"[{done}/{total}] {len(rows_mp)} rows, {(time.time()-t0)/60:.1f} min")
            ai.free_cuda()
print(f"done: {len(rows_mp)} rows in {(time.time()-t0)/60:.1f} min "
      f"({1 + len(lens_perms)} lenses x {total} forward passes)")

ai.save_json({"rows": rows_mp, "cfg": CFG, "targets": TARGETS, "controls": CONTROLS,
              "perm_seeds": PERM_SEEDS},
             "04k_c2_multicontrol_permutation_null.json")
print("saved -> 04k_c2_multicontrol_permutation_null.json")

[20/168] 5040 rows, 0.6 min
[40/168] 10080 rows, 1.3 min
[60/168] 15120 rows, 1.9 min
[80/168] 20160 rows, 2.6 min
[100/168] 25200 rows, 3.3 min
[120/168] 30240 rows, 3.9 min
[140/168] 35280 rows, 4.6 min
[160/168] 40320 rows, 5.2 min
done: 42336 rows in 5.5 min (21 lenses x 168 forward passes)
saved -> 04k_c2_multicontrol_permutation_null.json


In [11]:
def multicontrol_fold_by_lens(df):
    """multicontrol_stat's fold-change, computed separately for every lens_id present."""
    out = {}
    for lens_id, sub in df.groupby("lens_id"):
        stat = multicontrol_stat(
            sub.rename(columns={"p_target": "p_target_x"}), "p_target_x"
        )
        out[lens_id] = {r["layer"]: r["fold"] for r in stat}
    return out

df_mp = pd.DataFrame(rows_mp)
by_lens = multicontrol_fold_by_lens(df_mp)

print(f"{'layer':>5} {'real fold':>10} {'null range (20 perms)':>26} "
      f"{'perm p':>8} {'real rank in null':>18}")
perm_results = []
for L in LAYERS:
    real_fold = by_lens["real"][L]
    null_folds = sorted(by_lens[f"perm_{i:02d}"][L] for i in range(len(lens_perms)))
    real_log = abs(np.log(real_fold))
    as_extreme = sum(1 for f in null_folds if abs(np.log(f)) >= real_log)
    perm_p = (1 + as_extreme) / (1 + len(null_folds))
    rank = sum(1 for f in null_folds if f < real_fold)  # how many perms fall below real
    print(f"{L:>5} {real_fold:>9.3f}x   "
          f"[{min(null_folds):.3f}, {max(null_folds):.3f}]{'':>10} "
          f"{perm_p:>7.3f}   {rank}/{len(null_folds)} perms below real")
    perm_results.append({"layer": L, "real_fold": real_fold, "null_folds": null_folds,
                          "permutation_p": perm_p, "rank_of_real": rank})

ai.save_json({"perm_seeds": PERM_SEEDS, "results": perm_results},
             "04k_c2_multicontrol_permutation_stats.json")
print("saved -> 04k_c2_multicontrol_permutation_stats.json")

layer  real fold      null range (20 perms)   perm p  real rank in null
    9     0.921x   [0.962, 1.025]             0.048   0/20 perms below real
   18     1.057x   [0.918, 1.114]             0.238   18/20 perms below real
   27     0.863x   [0.911, 1.105]             0.048   0/20 perms below real
saved -> 04k_c2_multicontrol_permutation_stats.json


### Reading this result

- **Real fold sits inside the null range, perm p large** -> layer 27's negative result is
  indistinguishable from what an arbitrary lens permutation produces. The single-shuffle
  "SURVIVES" verdict above was the unlucky-draw case; the statistic itself does not carry
  lens-specific information. RULER's layer-27 result stands uncontested.
- **Real fold sits outside the null range, perm p small (<0.05)** -> the real lens
  genuinely produces a more extreme result than 20 independent permutations, in either
  direction. If negative and extreme, this is a real, lens-dependent, opposite-signed
  effect from RULER's -- goes in the packet as its own open question, not resolved by
  preference for either cohort.

## Content swap at matched length — answers Gautam's #1 and #3 in one sweep

Gautam, 8 Sep: *"I'd do the length-matched control first, since that directly tests whether
the cohort/construction difference is actually driving the result... I'd also prioritize
understanding why the layer-27 sign flips between digits and common-word needles."*

Those are the same experiment. The confound between RULER and the homemade cohort has three
strands tangled together — **construction** (RULER's context/question vs `build_niah_prompt`),
**length** (~15.7K vs a 64–8192 distance sweep), and **content** (7-digit numbers vs common
words). This holds construction and length fixed and varies only content:

|  | word needles | digit needles |
|---|---|---|
| homemade construction, distance 8192 (~16.4K tokens) | **this sweep** | **this sweep** |
| RULER construction, ~15.7K tokens | not buildable cheaply | already have it (run 026, positive) |

Distance 8192 gives `8064 + 128 + 8192 = 16,384` tokens against RULER's 15,679 — already
length-matched, so length is controlled by construction rather than by post-hoc weighting.
Distance 4096 is included as a second point to show whether the effect moves with length
at all.

**How to read it:**
- **Digit needles go POSITIVE, words stay negative** → content is the driver. Layer 27
  retrieves digit sequences and suppresses common words, in the same construction, at the
  same length. Gautam's #3 answered; RULER-vs-homemade is a content difference, not a
  cohort artefact.
- **Both stay negative** → construction is the driver, not content. RULER's positive result
  comes from something about its prompt format, and the "content-dependence" framing in the
  7 Sep findings entry is wrong and should be withdrawn.

**Scoring caveat, stated up front.** Word needles are single tokens (` Paris`), so the
readout at `pos=-1` scores them directly, as in Sơn's design. Digit strings tokenize with a
bare leading space (` 7700828` → `[' ', '7', '7', ...]`), so for the digit condition a space
is appended to the prompt and the first *digit* is scored. Each condition is internally
consistent and multi-control corrected within itself; the comparison is between the **signs
of two corrected effects**, not between raw magnitudes.

In [6]:
# Content swap: same construction, same length, word vs digit needles.
import numpy as np, pandas as pd, time
from scipy import stats as _stats

WORD_TARGETS  = ["Paris", "Tokyo", "banana", "lantern", "violin", "cinnamon", "harbour", "sapphire"]
WORD_CONTROLS = ["river", "chair", "window", "garden"]

# Fixed 7-digit strings, same shape as RULER's answers. Seeded so this is reproducible.
_rng = np.random.default_rng(ai.SEED)
_digits = ["".join(str(d) for d in _rng.integers(0, 10, size=7)) for _ in range(12)]
DIGIT_TARGETS, DIGIT_CONTROLS = _digits[:8], _digits[8:]

SWAP_DISTANCES = [4096, 8192]          # 8192 ~= 16.4K tokens, matched to RULER's 15.7K
SWAP_FILLERS   = range(EXP["n_filler_variants"])
SWAP_LAYERS    = EXP["layers"]

# word needles must be single-token for the pos=-1 readout to be well defined
WORD_TARGETS = [w for w in WORD_TARGETS
                if len(tok.encode(f" {w}", add_special_tokens=False)) == 1]
WORD_CONTROLS = [w for w in WORD_CONTROLS
                 if len(tok.encode(f" {w}", add_special_tokens=False)) == 1]
word_ids  = {w: tok.encode(f" {w}", add_special_tokens=False)[0] for w in WORD_TARGETS}
digit_ids = {str(d): tok.encode(str(d), add_special_tokens=False)[0] for d in range(10)}
assert len(set(digit_ids.values())) == 10

print(f"word targets ({len(WORD_TARGETS)}): {WORD_TARGETS}")
print(f"word controls ({len(WORD_CONTROLS)}): {WORD_CONTROLS}")
print(f"digit targets: {DIGIT_TARGETS}")
print(f"digit controls: {DIGIT_CONTROLS}")
n_pass = (len(WORD_TARGETS)+len(WORD_CONTROLS)+len(DIGIT_TARGETS)+len(DIGIT_CONTROLS)) \
         * len(SWAP_DISTANCES) * len(list(SWAP_FILLERS))
print(f"\nforward passes: {n_pass}")

word targets (7): ['Paris', 'Tokyo', 'banana', 'lantern', 'violin', 'cinnamon', 'harbour']
word controls (4): ['river', 'chair', 'window', 'garden']
digit targets: ['8300837', '7800236', '1041105', '6321504', '8932797', '9733910', '8365575', '8295196']
digit controls: ['9812529', '8230593', '5577316', '8517579']

forward passes: 138


In [ ]:
@torch.no_grad()
def score_swap(stored, content, distance, filler_idx):
    """One forward pass -> readout probability of every target of this content type.

    content="word":  score the single-token needle at pos=-1 (Sơn's design).
    content="digit": append a space so the next token is the first DIGIT, then score it.
    """
    spec = ai.build_niah_prompt(tok, stored, bundle, eviction_distance=distance,
                                 in_window=False, filler_idx=filler_idx)
    if not spec["needle_is_evicted"]:
        return []
    prompt = spec["prompt"] + (" " if content == "digit" else "")
    ins = tok(prompt, return_tensors="pt").to(bundle.model.device)

    try:
        on = probe.run(ins, nowrite=False, layers=SWAP_LAYERS, capture_residual=False)
    except torch.cuda.OutOfMemoryError as e:
        ai.free_cuda(); print(f"  ! OOM {stored}/{content}/d{distance}: {e}"); return []

    targets = WORD_TARGETS if content == "word" else DIGIT_TARGETS
    out = []
    for L in SWAP_LAYERS:
        lg = ai.readout_logits(on.o_t(L, pos=-1), bundle, lens=lens, layer=L)
        for t in targets:
            tid = word_ids[t] if content == "word" else digit_ids[t[0]]
            out.append({
                "stored": stored, "content": content, "tested_target": t,
                "is_target": stored in (WORD_TARGETS if content == "word" else DIGIT_TARGETS),
                "layer": L, "requested_distance": distance,
                "actual_eviction_distance": spec["actual_eviction_distance"],
                "n_tokens": spec["n_tokens"], "filler_idx": filler_idx,
                "p_target": ai.token_prob(lg, tid),
            })
    return out

swap_rows, t0 = [], time.time()
jobs = ([(w, "word") for w in WORD_TARGETS + WORD_CONTROLS]
        + [(d, "digit") for d in DIGIT_TARGETS + DIGIT_CONTROLS])
done = 0
for stored, content in jobs:
    for distance in SWAP_DISTANCES:
        for filler_idx in SWAP_FILLERS:
            swap_rows += score_swap(stored, content, distance, filler_idx)
            done += 1
            if done % 20 == 0:
                print(f"[{done}/{len(jobs)*len(SWAP_DISTANCES)*len(list(SWAP_FILLERS))}] "
                      f"{len(swap_rows)} rows, {(time.time()-t0)/60:.1f} min")
            ai.free_cuda()

print(f"done: {len(swap_rows)} rows in {(time.time()-t0)/60:.1f} min")
ai.save_json({"rows": swap_rows, "cfg": CFG, "exp": EXP,
              "word_targets": WORD_TARGETS, "word_controls": WORD_CONTROLS,
              "digit_targets": DIGIT_TARGETS, "digit_controls": DIGIT_CONTROLS,
              "distances": SWAP_DISTANCES},
             "04l_content_swap_rows.json")
print("saved -> 04l_content_swap_rows.json")

[20/138] 420 rows, 1.2 min
[40/138] 840 rows, 2.3 min
[60/138] 1260 rows, 3.6 min
[80/138] 1722 rows, 4.6 min
[100/138] 2202 rows, 5.7 min
[120/138] 2682 rows, 7.1 min
  ! OOM 8230593/digit/d8192: CUDA out of memory. Tried to allocate 346.00 MiB. GPU 0 has a total capacity of 19.62 GiB of which 18.94 MiB is free. Process 3308625 has 16.30 GiB memory in use. Process 3446999 has 18.78 GiB memory in use. Including non-PyTorch memory, this process has 8.60 GiB memory in use. Process 3465457 has 10.98 GiB memory in use. Of the allocated memory 7.48 GiB is allocated by PyTorch, and 926.78 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


In [ ]:
df_swap = pd.DataFrame(swap_rows)

def swap_stat(df, content, layer, distance=None):
    """Multi-control corrected effect, aggregated per target (targets are the sampling
    unit for a claim about content), bootstrapped over targets."""
    d = df[(df["content"] == content) & (df["layer"] == layer)]
    if distance is not None:
        d = d[d["requested_distance"] == distance]
    orig = d[(d["is_target"]) & (d["stored"] == d["tested_target"])][
        ["tested_target", "requested_distance", "filler_idx", "p_target"]
    ].rename(columns={"p_target": "p_own"})
    ctrl = d[~d["is_target"]].copy()
    ctrl["log_p"] = np.log(ctrl["p_target"] + 1e-30)
    cmean = (ctrl.groupby(["tested_target", "requested_distance", "filler_idx"])["log_p"]
                 .mean().reset_index(name="log_p_ctrl"))
    m = orig.merge(cmean, on=["tested_target", "requested_distance", "filler_idx"])
    m["delta_log"] = np.log(m["p_own"] + 1e-30) - m["log_p_ctrl"]
    per_target = m.groupby("tested_target")["delta_log"].mean()
    if len(per_target) < 3:
        return None
    xs = per_target.to_numpy()
    rng = np.random.default_rng(ai.SEED)
    reps = np.array([np.mean(rng.choice(xs, len(xs), replace=True)) for _ in range(10000)])
    return {"content": content, "layer": int(layer),
            "distance": "pooled" if distance is None else int(distance),
            "n_targets": len(xs), "fold": float(np.exp(xs.mean())),
            "ci_low": float(np.exp(np.percentile(reps, 2.5))),
            "ci_high": float(np.exp(np.percentile(reps, 97.5))),
            "excludes_null": bool(np.percentile(reps, 2.5) > 0 or np.percentile(reps, 97.5) < 0)}

results = []
print(f"{'layer':>5} {'content':>7} {'distance':>9} {'fold':>8} {'95% CI':>20} {'verdict':>12}")
for L in SWAP_LAYERS:
    for content in ("word", "digit"):
        for dist in SWAP_DISTANCES + [None]:
            r = swap_stat(df_swap, content, L, dist)
            if r is None: continue
            results.append(r)
            v = ("POSITIVE" if r["excludes_null"] and r["fold"] > 1 else
                 "NEGATIVE" if r["excludes_null"] and r["fold"] < 1 else "null")
            print(f"{L:>5} {content:>7} {str(r['distance']):>9} {r['fold']:>7.3f}x "
                  f"[{r['ci_low']:.3f}, {r['ci_high']:.3f}] {v:>12}")

ai.save_json({"results": results}, "04l_content_swap_stats.json")
print("\nsaved -> 04l_content_swap_stats.json")
print("\nLayer 27, matched length (distance 8192) is the row that answers Gautam:")
for r in results:
    if r["layer"] == 27 and r["distance"] == 8192:
        print(f"  {r['content']:>6}: {r['fold']:.3f}x [{r['ci_low']:.3f}, {r['ci_high']:.3f}]"
              f"  excludes null: {r['excludes_null']}")

In [ ]:
swap_rows, t0 = [], time.time()
jobs = ([(w, "word") for w in WORD_TARGETS + WORD_CONTROLS]
        + [(d, "digit") for d in DIGIT_TARGETS + DIGIT_CONTROLS])
total = len(jobs) * len(SWAP_DISTANCES) * len(list(SWAP_FILLERS))
done = 0
for stored, content in jobs:
    for distance in SWAP_DISTANCES:
        for filler_idx in SWAP_FILLERS:
            got = score_swap(stored, content, distance, filler_idx)
            if not got:
                print(f"  empty: {stored}/{content}/d{distance}/f{filler_idx}")
            swap_rows += got
            done += 1
            if done % 20 == 0:
                print(f"[{done}/{total}] {len(swap_rows)} rows, {(time.time()-t0)/60:.1f} min")
            ai.free_cuda()

print(f"done: {len(swap_rows)} rows in {(time.time()-t0)/60:.1f} min")
assert swap_rows, "sweep produced no rows — do not run stats, paste the 'empty:' lines above"
ai.save_json({"rows": swap_rows, "cfg": CFG, "exp": EXP,
              "word_targets": WORD_TARGETS, "word_controls": WORD_CONTROLS,
              "digit_targets": DIGIT_TARGETS, "digit_controls": DIGIT_CONTROLS,
              "distances": SWAP_DISTANCES},
             "04l_content_swap_rows.json")
print("saved -> 04l_content_swap_rows.json")